In [ ]:
import os
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models

from PIL import Image, ImageDraw

# Pointing to the directory containing dataset.py
sys.path.append('..')
from dataset import TimepieceDataset

compute_device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Active Compute Device: {compute_device}")

In [ ]:
class DigitalTimeReader(nn.Module):
    """
    ResNet18 backbone modified for triple-head classification:
      - fc_hour:   24 classes (0–23)
      - fc_minute: 60 classes (0–59)
      - fc_second: 60 classes (0–59)
    """
    def __init__(self):
        super().__init__()
        resnet_base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_feat = resnet_base.fc.in_features          
        resnet_base.fc = nn.Identity()             
        self.feature_extractor = resnet_base

        self.shared_mlp = nn.Sequential(
            nn.Linear(in_feat, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.fc_hour   = nn.Linear(256, 24)
        self.fc_minute = nn.Linear(256, 60)
        self.fc_second = nn.Linear(256, 60)

    def forward(self, x):
        features = self.feature_extractor(x)
        shared_out = self.shared_mlp(features)
        return self.fc_hour(shared_out), self.fc_minute(shared_out), self.fc_second(shared_out)

    def extract_time(self, x):
        out_h, out_m, out_s = self.forward(x)
        pred_h = out_h.argmax(dim=1)
        pred_m = out_m.argmax(dim=1)
        pred_s = out_s.argmax(dim=1)
        return pred_h, pred_m, pred_s

In [ ]:
# ===================== CELL 3: ERASER MODEL =====================
class DualConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)
    
class ClockFaceCleaner(nn.Module):
    """
    U-Net architecture mapped for removing clock hands.
    Generates a clean clock face tensor bounded in [0, 1].
    """
    def __init__(self, start_filters=64):
        super().__init__()
        
        self.down1 = DualConv(3, start_filters)
        self.down2 = DualConv(start_filters, start_filters*2)
        self.down3 = DualConv(start_filters*2, start_filters*4)
        self.down4 = DualConv(start_filters*4, start_filters*8)
        self.max_pool = nn.MaxPool2d(2, 2)

        self.bridge = DualConv(start_filters*8, start_filters*8)

        self.upconv4 = nn.ConvTranspose2d(start_filters*8, start_filters*8, kernel_size=2, stride=2)
        self.up4 = DualConv(start_filters*8 + start_filters*8, start_filters*4)

        self.upconv3 = nn.ConvTranspose2d(start_filters*4, start_filters*4, kernel_size=2, stride=2)
        self.up3 = DualConv(start_filters*4 + start_filters*4, start_filters*2)

        self.upconv2 = nn.ConvTranspose2d(start_filters*2, start_filters*2, kernel_size=2, stride=2)
        self.up2 = DualConv(start_filters*2 + start_filters*2, start_filters)

        self.upconv1 = nn.ConvTranspose2d(start_filters, start_filters, kernel_size=2, stride=2)
        self.up1 = DualConv(start_filters + start_filters, start_filters)

        self.out_conv = nn.Sequential(
            nn.Conv2d(start_filters, 3, kernel_size=1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.max_pool(d1))
        d3 = self.down3(self.max_pool(d2))
        d4 = self.down4(self.max_pool(d3))
        
        mid = self.bridge(self.max_pool(d4))

        u4 = self.up4(torch.cat([self._align(self.upconv4, mid, d4), d4], dim=1))
        u3 = self.up3(torch.cat([self._align(self.upconv3, u4, d3), d3], dim=1))
        u2 = self.up2(torch.cat([self._align(self.upconv2, u3, d2), d2], dim=1))
        u1 = self.up1(torch.cat([self._align(self.upconv1, u2, d1), d1], dim=1))

        return self.out_conv(u1)

    @staticmethod
    def _align(up_layer, src, target):
        src = up_layer(src)
        if src.shape != target.shape:
            src = F.interpolate(src, size=target.shape[2:], mode='bilinear', align_corners=False)
        return src

In [ ]:
def render_hands_on_faces(cleared_batch, hr_tensors, min_tensors, sec_tensors):
    """Projects pixel-perfect hands onto the cleaned U-Net outputs."""
    b_size, ch, height, width = cleared_batch.shape
    rendered_images = []

    for idx in range(b_size):
        bg_face = cleared_batch[idx].clone()
        val_h = int(hr_tensors[idx].item())
        val_m = int(min_tensors[idx].item())
        val_s = int(sec_tensors[idx].item())

        img_np = (bg_face.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')
        img_pil = Image.fromarray(img_np)
        canvas = ImageDraw.Draw(img_pil)

        center_x, center_y = width / 2, height / 2
        max_rad = min(width, height) / 2 - max(5, width // 25)

        rad_h = math.radians((val_h % 12) * 30 + val_m * 0.5 - 90)
        rad_m = math.radians(val_m * 6 + val_s * 0.1 - 90)
        rad_s = math.radians(val_s * 6 - 90)

        def get_coords(theta, length):
            return (center_x + length * math.cos(theta),
                    center_y + length * math.sin(theta))

        px_hx, px_hy = get_coords(rad_h, max_rad * 0.50)
        canvas.line([center_x, center_y, px_hx, px_hy], fill=(30, 30, 30), width=max(3, int(width * 0.04)))

        px_mx, px_my = get_coords(rad_m, max_rad * 0.75)
        canvas.line([center_x, center_y, px_mx, px_my], fill=(30, 30, 30), width=max(2, int(width * 0.025)))

        px_sx, px_sy = get_coords(rad_s, max_rad * 0.85)
        canvas.line([center_x, center_y, px_sx, px_sy], fill=(200, 40, 40), width=max(1, int(width * 0.008)))

        cap_size = max(3, width // 40)
        canvas.ellipse([center_x - cap_size, center_y - cap_size, center_x + cap_size, center_y + cap_size], fill=(30, 30, 30))

        tensor_out = torch.from_numpy(np.array(img_pil).astype('float32') / 255.0).permute(2, 0, 1)
        rendered_images.append(tensor_out)

    return torch.stack(rendered_images)

In [ ]:
WEIGHTS_READER = 'checkpoints/digital_reader_best.pth'
WEIGHTS_ERASER = 'checkpoints/eraser_v2_best.pth'

time_reader = DigitalTimeReader().to(compute_device)
time_reader.load_state_dict(torch.load(WEIGHTS_READER, map_location=compute_device))
time_reader.eval()
print("Reader initialized ✓")

face_cleaner = ClockFaceCleaner().to(compute_device)
face_cleaner.load_state_dict(torch.load(WEIGHTS_ERASER, map_location=compute_device))
face_cleaner.eval()
print("Cleaner initialized ✓")

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

reader_preprocessing = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Implemented the dataset class signature changes here
dataset_obj = TimepieceDataset(base_path="..\datasets", partition='test', transforms_pipeline=base_transforms)
data_loader = DataLoader(dataset_obj, batch_size=6, shuffle=True, num_workers=0)
sample_batch = next(iter(data_loader))

digital_reader_inputs = torch.stack([
    reader_preprocessing(
        transforms.ToPILImage()(sample_batch['digital_img'][idx])
    ) for idx in range(len(sample_batch['digital_img']))
]).to(compute_device)

analog_inputs = sample_batch['analog_img'].to(compute_device)
ground_truths = sample_batch['original_time']

with torch.no_grad():
    pred_h, pred_m, pred_s = time_reader.extract_time(digital_reader_inputs)
    cleaned_analog_faces = face_cleaner(analog_inputs)
    final_visuals = render_hands_on_faces(cleaned_analog_faces.cpu(), pred_h.cpu(), pred_m.cpu(), pred_s.cpu())

# --- Plotting ---
batch_size_n = len(sample_batch['digital_img'])
figure, axs = plt.subplots(4, batch_size_n, figsize=(3 * batch_size_n, 12))
plot_titles = ['Digital In', 'Analog In', 'Generated Out', 'Target (GT)']

for col_idx in range(batch_size_n):
    t_h = int(ground_truths[col_idx][0])
    t_m = int(ground_truths[col_idx][1])
    t_s = int(ground_truths[col_idx][2])
    
    p_h = int(pred_h[col_idx].item())
    p_m = int(pred_m[col_idx].item())
    p_s = int(pred_s[col_idx].item())
    is_match = (t_h == p_h and t_m == p_m and t_s == p_s)

    axs[0, col_idx].imshow(sample_batch['digital_img'][col_idx].permute(1, 2, 0))
    axs[0, col_idx].set_title(
        f"predicted: {p_h:02d}:{p_m:02d}:{p_s:02d}",
        color='green' if is_match else 'red', fontsize=8
    )

    axs[1, col_idx].imshow(analog_inputs[col_idx].cpu().permute(1, 2, 0))
    axs[1, col_idx].set_title(f"actual: {t_h:02d}:{t_m:02d}:{t_s:02d}", fontsize=8)

    axs[2, col_idx].imshow(final_visuals[col_idx].permute(1, 2, 0).clamp(0, 1))
    axs[2, col_idx].set_title("drawn output", fontsize=8)

    axs[3, col_idx].imshow(sample_batch['analog_img'][col_idx].permute(1, 2, 0))
    axs[3, col_idx].set_title("clean ground truth", fontsize=8)

    for row_idx in range(4):
        if col_idx == 0:
            axs[row_idx, 0].set_ylabel(plot_titles[row_idx], fontsize=9, rotation=90)
        axs[row_idx, col_idx].axis('off')

plt.suptitle("End-to-End Pipeline: Digital to Analog", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()